In [20]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
from scipy.signal import butter, filtfilt
import pandas as pd
import neurokit2 as nk

# Parameters (adjust if needed)
pixels_per_mm = 10  # adjust manually if needed
paper_speed = 25.0  # mm/s
desired_fs = 500    # Hz

def preprocess_ecg(img, clahe=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if clahe:
        clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray = clahe_obj.apply(gray)
    gray = cv2.medianBlur(gray, 3)
    return gray

def extract_waveform(gray_img):
    thresh = threshold_otsu(gray_img)
    bw = gray_img < thresh  # waveform is darker
    skeleton = skeletonize(bw)
    xs, ys = [], []
    h, w = skeleton.shape
    for x in range(w):
        rows = np.where(skeleton[:, x])[0]
        if len(rows) == 0:
            continue
        y_med = int(np.median(rows))
        xs.append(x)
        ys.append(y_med)
    return np.array(xs), np.array(ys), skeleton

def pixels_to_signal(xs, ys, pixels_per_mm, paper_speed=25.0, desired_fs=500):
    fs_est = pixels_per_mm * paper_speed
    dt = 1.0 / fs_est
    t_pixels = xs * dt
    baseline = np.median(ys)
    mV_per_pixel = 1.0 / (10.0 * pixels_per_mm)
    amps_mV = (baseline - ys) * mV_per_pixel
    t_uniform = np.arange(t_pixels.min(), t_pixels.max(), 1.0/desired_fs)
    amps_uniform = np.interp(t_uniform, t_pixels, amps_mV)
    return t_uniform, amps_uniform, fs_est

def butter_bandpass(lowcut, highcut, fs, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return b, a

def filter_ecg(signal, fs, lowcut=0.5, highcut=40.0):
    b, a = butter_bandpass(lowcut, highcut, fs)
    filtered = filtfilt(b, a, signal)
    return filtered

def process_ecg_image(img_path):
    img = cv2.imread(img_path)
    gray_img = preprocess_ecg(img)
    xs, ys, skeleton = extract_waveform(gray_img)
    t, ecg_signal, fs = pixels_to_signal(xs, ys, pixels_per_mm, paper_speed, desired_fs)
    ecg_filtered = filter_ecg(ecg_signal, fs)
    ecg_cleaned = nk.ecg_clean(ecg_filtered, sampling_rate=fs)
    signals, info = nk.ecg_peaks(ecg_cleaned, sampling_rate=fs)
    r_peaks = info['ECG_R_Peaks']
    rr_intervals = np.diff(t[r_peaks]) if len(r_peaks) > 1 else np.array([])
    heart_rate = 60 / rr_intervals if rr_intervals.size > 0 else np.array([])

    hrv_features = pd.DataFrame()
    if len(r_peaks) > 1 and rr_intervals.size > 1 and not np.any(np.isnan(rr_intervals)):
        try:
            hrv_features = nk.hrv(info, sampling_rate=fs, show=False)
        except Exception as e:
            print(f"HRV calculation failed for {img_path}: {e}")
            hrv_features = pd.DataFrame()

    return {
        'image_path': img_path,
        'r_peaks_count': len(r_peaks),
        'mean_hr': np.mean(heart_rate) if heart_rate.size > 0 else None,
        'rr_intervals': rr_intervals,
        'hrv_features': hrv_features
    }

# Directory path containing ECG images
folder_path = r"C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)"  # change to your folder path

# List all image files in the folder
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

results = []
for image_file in image_files:
    full_path = os.path.join(folder_path, image_file)
    result = process_ecg_image(full_path)
    results.append(result)

# Combine HRV features for all processed images into one DataFrame
all_hrv_features = pd.concat([r['hrv_features'].assign(image=r['image_path']) for r in results if not r['hrv_features'].empty], ignore_index=True)

# Save combined HRV features to CSV
all_hrv_features.to_csv('ecg_features_all_images.csv', index=False)

# Print a summary of results
for r in results:
    print(f"Image: {r['image_path']}, R-peaks: {r['r_peaks_count']}, Mean HR: {r['mean_hr']}")


HRV calculation failed for C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(1).jpg: SVD did not converge in Linear Least Squares
HRV calculation failed for C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(101).jpg: SVD did not converge in Linear Least Squares
HRV calculation failed for C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(103).jpg: SVD did not converge in Linear Least Squares
HRV calculation failed for C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(104).jpg: SVD did not converge in Linear Least Squares
HRV calculation failed for C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(105).jpg: SVD did not converge in Linear Least Squares
HRV calculation failed for C:\Users\shiva\major project\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(107).jpg: SVD did n

KeyboardInterrupt: 